# 04 — Validación del Motor de Rutas de Emergencia

Validación end-to-end del sistema completo:
1. Carga del grafo de Madrid
2. Cálculo de rutas desde parques de bomberos reales a puntos de incidente
3. Comparativa con/sin predicción de tráfico
4. Análisis de impacto de las restricciones de anchura
5. Visualización de rutas sobre el mapa

In [ ]:
import sys
sys.path.insert(0, '..')

import os
import time
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import folium

from routing.graph_engine import load_graph, get_nearest_node
from routing.optimizer import calcular_ruta, ANCHO_CAMION_REQ
from ml.predict import predecir_trafico
from ml.generate_dataset import DISTRITOS_MADRID

## 1. Carga del grafo

In [ ]:
t0 = time.time()
G = load_graph()
print(f'Grafo cargado en {time.time() - t0:.1f}s')
print(f'Nodos: {G.number_of_nodes():,}  |  Aristas: {G.number_of_edges():,}')
print(f'Anchura mínima requerida: {ANCHO_CAMION_REQ}m')

## 2. Definición de escenarios de prueba

Parques de bomberos reales de Madrid y puntos de incidente representativos.

In [ ]:
# Coordenadas aproximadas de parques de bomberos de Madrid
PARQUES = {
    'P1 - Retiro':        (40.4168, -3.6792),
    'P2 - Vallecas':      (40.3919, -3.6673),
    'P3 - Latina':        (40.4075, -3.7208),
    'P4 - Hortaleza':     (40.4700, -3.6380),
    'P5 - Carabanchel':   (40.3820, -3.7380),
}

# Puntos de incidente de prueba (representan distintas zonas)
INCIDENTES = {
    'Gran Vía':           (40.4200, -3.7050),
    'Aeropuerto Barajas': (40.4936, -3.5668),
    'Vallecas':           (40.3868, -3.6545),
    'Plaza España':       (40.4240, -3.7122),
    'Legazpi':            (40.3948, -3.6963),
}

print(f'{len(PARQUES)} parques × {len(INCIDENTES)} incidentes = {len(PARQUES)*len(INCIDENTES)} rutas a calcular')

## 3. Cálculo de rutas — comparativa con/sin tráfico

In [ ]:
FECHA_TEST = '2025-06-16'   # lunes laborable
HORA_TEST = 8               # hora punta mañana

traffic_preds = predecir_trafico(FECHA_TEST, HORA_TEST, DISTRITOS_MADRID)
print(f'Predicción de tráfico para {FECHA_TEST} {HORA_TEST}:00h')
for d, n in sorted(traffic_preds.items(), key=lambda x: -x[1]):
    label = {0: 'Bajo', 1: 'Medio', 2: 'Alto'}[n]
    print(f'  {d:30s}: {label}')

In [ ]:
resultados = []

for nombre_parque, (p_lat, p_lon) in PARQUES.items():
    for nombre_inc, (i_lat, i_lon) in INCIDENTES.items():
        orig = get_nearest_node(p_lat, p_lon)
        dest = get_nearest_node(i_lat, i_lon)

        # Sin tráfico
        r_sin = calcular_ruta(G, orig, dest, {})
        # Con tráfico
        r_con = calcular_ruta(G, orig, dest, traffic_preds)

        resultados.append({
            'parque': nombre_parque,
            'incidente': nombre_inc,
            'dist_m': r_sin['features'][0]['properties']['length_m'] if r_sin else None,
            'tiempo_sin_trafico_s': r_sin['features'][0]['properties']['time_s'] if r_sin else None,
            'tiempo_con_trafico_s': r_con['features'][0]['properties']['time_s'] if r_con else None,
            'ruta_existe': r_sin is not None,
        })

df_res = pd.DataFrame(resultados)
df_res['tiempo_sin_min'] = df_res['tiempo_sin_trafico_s'] / 60
df_res['tiempo_con_min'] = df_res['tiempo_con_trafico_s'] / 60
df_res['penalizacion_min'] = df_res['tiempo_con_min'] - df_res['tiempo_sin_min']

df_res[['parque', 'incidente', 'dist_m', 'tiempo_sin_min', 'tiempo_con_min', 'penalizacion_min']].round(2)

## 4. Análisis de resultados

In [ ]:
df_validas = df_res[df_res['ruta_existe']]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Distribución de tiempos
df_validas['tiempo_sin_min'].hist(bins=15, ax=axes[0], color='#22c55e', edgecolor='white', alpha=0.8, label='Sin tráfico')
df_validas['tiempo_con_min'].hist(bins=15, ax=axes[0], color='#ef4444', edgecolor='white', alpha=0.6, label='Con tráfico')
axes[0].set_title('Distribución de tiempos de respuesta')
axes[0].set_xlabel('Tiempo (min)')
axes[0].legend()

# Penalización por tráfico
df_validas['penalizacion_min'].hist(bins=15, ax=axes[1], color='#f59e0b', edgecolor='white')
axes[1].set_title('Penalización por tráfico')
axes[1].set_xlabel('Minutos adicionales')

# Tiempo medio por parque
df_validas.groupby('parque')['tiempo_con_min'].mean().sort_values().plot(kind='barh', ax=axes[2], color='steelblue', edgecolor='white')
axes[2].set_title('Tiempo medio de respuesta por parque')
axes[2].set_xlabel('Tiempo medio (min)')

plt.tight_layout()
plt.show()

print(f'\nResumen:')
print(f'  Rutas calculadas con éxito: {df_validas.shape[0]} / {len(df_res)}')
print(f'  Tiempo medio sin tráfico:   {df_validas["tiempo_sin_min"].mean():.1f} min')
print(f'  Tiempo medio con tráfico:   {df_validas["tiempo_con_min"].mean():.1f} min')
print(f'  Penalización media tráfico: +{df_validas["penalizacion_min"].mean():.1f} min')

## 5. Visualización interactiva de una ruta

In [ ]:
# Visualizar la ruta más larga encontrada
caso = df_validas.sort_values('tiempo_con_min', ascending=False).iloc[0]
print(f'Ruta seleccionada: {caso["parque"]} → {caso["incidente"]}')
print(f'Tiempo: {caso["tiempo_con_min"]:.1f} min | Distancia: {caso["dist_m"]/1000:.2f} km')

p_lat, p_lon = PARQUES[caso['parque']]
i_lat, i_lon = INCIDENTES[caso['incidente']]

orig = get_nearest_node(p_lat, p_lon)
dest = get_nearest_node(i_lat, i_lon)
ruta = calcular_ruta(G, orig, dest, traffic_preds)

if ruta:
    coords = ruta['features'][0]['geometry']['coordinates']  # [lon, lat]
    centro = [np.mean([c[1] for c in coords]), np.mean([c[0] for c in coords])]

    m = folium.Map(location=centro, zoom_start=13, tiles='CartoDB dark_matter')

    folium.Marker([p_lat, p_lon], popup=caso['parque'],
                  icon=folium.Icon(color='blue', icon='home', prefix='fa')).add_to(m)
    folium.Marker([i_lat, i_lon], popup='Incidente',
                  icon=folium.Icon(color='red', icon='fire', prefix='fa')).add_to(m)
    folium.PolyLine(
        [(c[1], c[0]) for c in coords],
        color='#f59e0b', weight=5, opacity=0.9,
        tooltip=f"{caso['dist_m']/1000:.2f} km — {caso['tiempo_con_min']:.1f} min"
    ).add_to(m)

    display(m)
else:
    print('No se pudo calcular la ruta para este caso.')

## 6. Impacto de las restricciones de anchura

In [ ]:
# Comparar número de aristas disponibles según el umbral de anchura
anchos_prueba = [2.0, 2.5, 3.0, 3.5, 4.0, 4.5]
total_aristas = G.number_of_edges()

disponibles = []
for umbral in anchos_prueba:
    n = sum(1 for u, v, d in G.edges(data=True) if d.get('width_m', 0) >= umbral)
    disponibles.append({'umbral_m': umbral, 'aristas': n, 'porcentaje': n / total_aristas * 100})

df_anchos = pd.DataFrame(disponibles)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(df_anchos['umbral_m'].astype(str), df_anchos['porcentaje'], color='steelblue', edgecolor='white')
ax.axvline(x=anchos_prueba.index(ANCHO_CAMION_REQ), color='red', linestyle='--', label=f'Config actual ({ANCHO_CAMION_REQ}m)')
ax.set_title('% de aristas accesibles según anchura mínima del vehículo')
ax.set_xlabel('Anchura mínima requerida (m)')
ax.set_ylabel('% aristas disponibles')
ax.legend()
plt.tight_layout()
plt.show()

print(df_anchos.to_string(index=False))